# Data ingest and data cleaning

In [82]:
# imports

import datetime as dt
from pathlib import Path

import kagglehub
import pandas as pd


In [ ]:
dataset_path = Path(
    kagglehub.dataset_download("mashlyn/online-retail-ii-uci")
)

files = list(dataset_path.iterdir())
print([file.name for file in files])

data_file = next(
    file for file in files
    if file.suffix.lower() in {".xlsx", ".xls", ".csv"}
)

df = pd.read_csv(data_file)

df.to_csv("../01_data/01_raw_data/customer_original.csv", index=False)

df

['online_retail_II.csv']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [91]:
# -----------
# DATA AUDIT
# -----------

df.shape

(1067371, 8)

In [92]:
df.dtypes

# Need new Format: InvoiceDate (date), Customer ID (str)

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

In [93]:
# New Format 'InvoiceDate'

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# New Format and Name 'Customer ID'

df["CustomerID"] = (
    pd.to_numeric(df["Customer ID"], errors="coerce")
    .astype("Int64")
)
df['CustomerID'] = df['CustomerID'].astype(str)

df = df.drop(columns=['Customer ID'])

df.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Country,CustomerID
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,United Kingdom,13085
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,United Kingdom,13085
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,United Kingdom,13085
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,United Kingdom,13085
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,United Kingdom,13085


In [94]:
df.isna().sum()

# Missing Values: Customer ID without ID, Description

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Country             0
CustomerID     243007
dtype: int64

In [95]:
df.describe()

# Negative values: Quantity, Price

,Quantity,InvoiceDate,Price
count,1.067371e+06,1067371,1.067371e+06
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04
std,1.727058e+02,NaN,1.235531e+02


In [96]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  str           
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Country      1067371 non-null  str           
 7   CustomerID   824364 non-null   str           
dtypes: datetime64[us](1), float64(1), int64(1), str(5)
memory usage: 65.1 MB


In [97]:
df.to_csv("../01_data/02_processed_data/customer.csv", index=False)